In [ ]:
# import paligemma from transformers
import torch
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from transformers.generation import StoppingCriteria, StoppingCriteriaList
from PIL import Image

MODEL_ID = "google/paligemma-3b-mix-448"
device = "cuda" if torch.cuda.is_available() else "cpu"


class MaxWordsStoppingCriteria(StoppingCriteria):
    """Stop generation once `max_words` whitespace-delimited words have been produced."""
    def __init__(self, max_words: int, tokenizer):
        self.max_words = max_words
        self.tokenizer = tokenizer

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # Decode only the newly generated tokens (skip the input prompt)
        # We rely on the fact that input_ids grows as generation proceeds
        decoded = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
        word_count = len(decoded.split())
        return word_count >= self.max_words

/home2/akash.manna/miniconda3/envs/datacomp/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 1.13.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
# load processor and model
processor = PaliGemmaProcessor.from_pretrained(MODEL_ID)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).to(device)
model.eval()

In [ ]:
# build the prompt: ground description with page title + alt-text
#ref: https://arxiv.org/pdf/2407.07726v1#page=13.69
alt_text = "REPLACE_WITH_ALT_TEXT"

prompt = (
    f"The image presented came from a web"
    f"and had the alt-text: {alt_text}. Please describe what is in the image "
    f"using the alt-text and the page title as a guide to ground your response. "
    f"For example, if the alt-text contains a specific brand name, use that brand "
    f"name in your output. Please be descriptive but concise. DO NOT make things up. "
    f"If you can't tell something with certainty in the image, simply don't say "
    f"anything about it.\ncaption en"
)

In [ ]:
# prepare input — load image and process with the prompt
image_path = "REPLACE_WITH_IMAGE_PATH"  # e.g., "/path/to/image.jpg"
raw_image = Image.open(image_path).convert("RGB")

inputs = processor(
    images=raw_image,
    text=prompt,
    return_tensors="pt",
).to(device)
# truncate if needed (PaliGemma has limited input length)
inputs = {k: v[:, :processor.image_seq_length + processor.text_seq_length] if len(v.shape) > 1 else v for k, v in inputs.items()}

In [ ]:
# generate caption tokens — stop after at most 10 words
MAX_WORDS = 10
stop_criteria = StoppingCriteriaList([MaxWordsStoppingCriteria(MAX_WORDS, processor.tokenizer)])

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=128,          # safety cap in tokens
        do_sample=False,
        num_beams=4,
        stopping_criteria=stop_criteria,
    )

In [ ]:
# decode the generated token ids to text
generated_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

In [ ]:
# clean up caption — strip the prompt prefix, keep only the generated response
caption = generated_text[len(prompt):].strip()
print("Generated caption:")
print(caption)

In [ ]:
import os
os._exit(0)

: 